# WOD-E2E Dataset EDA
**DATASCI 281 · Group 2**

This notebook carefully verifies every structural assumption about the dataset before
any feature extraction. It uses the **official proto** (`E2EDFrame`) rather than
raw JPEG byte-scanning, so what we read is exactly what Waymo intended.

### What the official proto tells us (from `end_to_end_driving_data.proto`)
Each TFRecord record deserializes to an **`E2EDFrame`** containing:
- `frame.context.name` — unique segment identifier (maps to `val_sequence_name_to_scenario_cluster.json`)
- `frame.timestamp_micros` — frame timestamp in **microseconds** since Unix epoch
- `frame.images[]` — list of `CameraImage` protos, one per camera (exactly **8 cameras**)  
  Names: `FRONT=1, FRONT_LEFT=2, FRONT_RIGHT=3, SIDE_LEFT=4, SIDE_RIGHT=5, REAR=6, REAR_LEFT=7, REAR_RIGHT=8`
- `frame.context.camera_calibrations[]` — intrinsics + extrinsics for all 8 cameras
- `past_states` — 4 s of ego trajectory at 4 Hz (16 waypoints)
- `future_states` — 5 s future trajectory (train/val only)
- `intent` — `GO_STRAIGHT | GO_LEFT | GO_RIGHT`
- `preference_trajectories` — rater feedback (val only, single frame per segment)

Dataset composition (from Waymo docs):
- **Training:** 2,037 segments × 20 s × 10 Hz = ~407k frames
- **Validation:** 479 segments × 20 s × 10 Hz = ~95k frames  
- **Testing:** 1,505 segments × 12 s × 10 Hz = ~181k frames

Val split has 93 tfrecord shards. We are working from shard 00.


## 0. Imports & Paths

In [ ]:
import json, io
from pathlib import Path
from collections import defaultdict, Counter
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

DATA_DIR    = Path("../data")
SHARD_PATH  = DATA_DIR / "val_202504211843.tfrecord-00000-of-00093"
LABELS_PATH = DATA_DIR / "val_sequence_name_to_scenario_cluster.json"

assert SHARD_PATH.exists(),  f"Missing: {SHARD_PATH}"
assert LABELS_PATH.exists(), f"Missing: {LABELS_PATH}"
print("Both files found.")
print(f"Shard size: {SHARD_PATH.stat().st_size / 1e9:.2f} GB")


## 1. Inspect the Labels File

Ground truth before we touch any images.  
**Note from Waymo docs:** the official scenario clusters include a `Spotlight` category
(manually selected challenging scenarios) in addition to the 10 listed in the challenge.


In [ ]:
with open(LABELS_PATH) as f:
    labels = json.load(f)

first_key = next(iter(labels))
print("Example key:", first_key)
print("Example value:", json.dumps(labels[first_key], indent=2))
print(f"\nTotal labeled sequences in val set: {len(labels):,}")
print("Fields per entry:", list(labels[first_key].keys()))


In [ ]:
# Full class distribution
cluster_counts = Counter(v["scenario_cluster"] for v in labels.values())
print(f"Distinct classes found: {len(cluster_counts)}\n")
print("Class distribution — full val label set:\n")
for cls, cnt in sorted(cluster_counts.items(), key=lambda x: -x[1]):
    pct = 100 * cnt / len(labels)
    bar = "\u2588" * int(pct / 2)
    print(f"  {cls:<32s}  {cnt:4d}  ({pct:5.1f}%)  {bar}")

# Note any classes beyond the 10 we expected
expected = {"Construction","Cut-ins","Cyclist","Foreign Object Debris",
            "Intersections","Multi-Lane Maneuvers","Others","Pedestrian",
            "Single-Lane Maneuvers","Special Vehicles"}
found = set(cluster_counts.keys())
print(f"\nExpected 10 classes. Found: {len(found)}")
print("Extra classes:", found - expected if found - expected else "(none)")
print("Missing classes:", expected - found if expected - found else "(none)")


In [ ]:
# Multi-label check
# The labels JSON maps one seq_id -> one scenario_cluster string.
# Multi-label would mean a seq_id appears MORE THAN ONCE in the JSON,
# OR scenario_cluster is a list rather than a string.
multi_list = {k: v for k, v in labels.items()
              if isinstance(v["scenario_cluster"], list)}
print(f"Entries where scenario_cluster is a list: {len(multi_list)}")

# Check for duplicate keys (JSON can't have these by spec, but verify)
print(f"Unique keys in JSON: {len(labels):,} (JSON keys are always unique by spec)")

print("\n→ Multi-label is NOT encoded in the labels JSON directly.")
print("  It would require a sequence to appear in MULTIPLE label files or")
print("  a separate multi-label JSON. Check if there are other label files in GCS.")


## 2. Parse the TFRecord Using the Official E2EDFrame Proto

We install `waymo-open-dataset` and use the actual proto parser.
This is strictly better than byte-scanning: it gives us named fields,
correct camera ordering, timestamps, and calibration data.

```bash
pip install waymo-open-dataset-tf-2-12-0
```
If that fails (e.g. ARM64 Mac), use the fallback in Section 2b.


In [ ]:
# Install if needed — comment out after first run
# !pip install waymo-open-dataset-tf-2-12-0 -q

try:
    from waymo_open_dataset.protos import end_to_end_driving_data_pb2 as e2e_pb2
    from waymo_open_dataset.protos import dataset_pb2
    PROTO_AVAILABLE = True
    print("waymo_open_dataset proto available ✓")
except ImportError as e:
    PROTO_AVAILABLE = False
    print(f"Proto not available: {e}")
    print("→ Will fall back to TF Example + JPEG scan (Section 2b)")


In [ ]:
import tensorflow as tf

if PROTO_AVAILABLE:
    # Parse using official E2EDFrame proto
    dataset = tf.data.TFRecordDataset(str(SHARD_PATH))

    print("Parsing first record with E2EDFrame proto...\n")
    for raw in dataset.take(1):
        frame_data = e2e_pb2.E2EDFrame()
        frame_data.ParseFromString(raw.numpy())

    # --- context.name (the sequence identifier) ---
    ctx_name = frame_data.frame.context.name
    print(f"frame.context.name: '{ctx_name}'")
    print(f"  In labels JSON? {ctx_name in labels}")
    if ctx_name in labels:
        print(f"  Cluster: {labels[ctx_name]['scenario_cluster']}")

    # --- timestamp ---
    ts_us = frame_data.frame.timestamp_micros
    import datetime
    dt = datetime.datetime.utcfromtimestamp(ts_us / 1e6)
    print(f"\nframe.timestamp_micros: {ts_us}")
    print(f"  As UTC datetime: {dt}")

    # --- cameras ---
    images = frame_data.frame.images
    print(f"\nframe.images count: {len(images)}")
    print(f"{'cam_name':>12}  {'name_int':>8}  {'JPEG bytes':>12}  {'width':>6}  {'height':>6}")
    print("-" * 55)
    for img in images:
        pil = Image.open(io.BytesIO(img.image))
        print(f"{dataset_pb2.CameraName.Name(img.name).name:>12}  {img.name:>8}  {len(img.image):>12,}  {pil.width:>6}  {pil.height:>6}")

    # --- intent ---
    intent_name = e2e_pb2.EgoIntent.Intent.Name(frame_data.intent)
    print(f"\nintent: {intent_name}")

    # --- past trajectory ---
    n_past = len(frame_data.past_states.pos_x)
    print(f"past_states waypoints: {n_past}  (expect 16 = 4s × 4Hz)")

    # --- future trajectory ---
    n_future = len(frame_data.future_states.pos_x)
    print(f"future_states waypoints: {n_future}  (expect 20 = 5s × 4Hz)")

    # --- calibrations ---
    n_cal = len(frame_data.frame.context.camera_calibrations)
    print(f"camera_calibrations: {n_cal}")


In [ ]:
# FALLBACK: If proto isn't available, parse via TF Example + JPEG scan
# This gives us fewer fields (no intent, no trajectory, no named cameras)
# but still lets us verify camera count and timestamps from the key.

if not PROTO_AVAILABLE:
    print("Using TF Example fallback...\n")
    JPEG_SOI = b'\xff\xd8\xff'
    JPEG_EOI = b'\xff\xd9'

    def find_all_jpegs(data):
        results, pos = [], 0
        while True:
            s = data.find(JPEG_SOI, pos)
            if s == -1: break
            e = data.find(JPEG_EOI, s)
            if e == -1: break
            results.append((s, e + 2))
            pos = e + 2
        return results

    dataset = tf.data.TFRecordDataset(str(SHARD_PATH))
    for raw in dataset.take(1):
        b = raw.numpy()
        ex = tf.train.Example()
        ex.ParseFromString(b)
        keys = list(ex.features.feature.keys())
        print(f"Feature keys: {keys}")
        jpegs = find_all_jpegs(b)
        print(f"JPEGs found by byte scan: {len(jpegs)}")
        print("\n⚠ This method does NOT give named cameras or timestamps.")
        print("  Camera order by position in bytes is NOT guaranteed to match")
        print("  the CameraName enum order (FRONT=1, FRONT_LEFT=2, ...).")
        print("  Install waymo-open-dataset for reliable camera identification.")


## 3. Sequence → Frame Structure

Scan the full shard to build `context.name → [timestamp_micros]`.  
This tells us sequences per shard, frames per sequence, and the time range.

**Note:** `frame.context.name` is the segment identifier, not the TFRecord file name.
Multiple frames within the same shard will share the same `context.name`.


In [ ]:
seq_to_timestamps = defaultdict(list)  # context.name -> list of timestamp_micros

print("Scanning full shard (this takes a minute)...")
n_records = 0

if PROTO_AVAILABLE:
    for raw in dataset:
        fd = e2e_pb2.E2EDFrame()
        fd.ParseFromString(raw.numpy())
        name = fd.frame.context.name
        ts   = fd.frame.timestamp_micros
        seq_to_timestamps[name].append(ts)
        n_records += 1
else:
    # Fallback: extract seq_id from TF key, no timestamps
    for raw in dataset:
        b = raw.numpy()
        ex = tf.train.Example()
        ex.ParseFromString(b)
        for k in ex.features.feature.keys():
            parts = k.rsplit("-", 1)
            if len(parts) == 2:
                seq_to_timestamps[parts[0]].append(None)
        n_records += 1

n_seqs   = len(seq_to_timestamps)
n_frames = sum(len(v) for v in seq_to_timestamps.values())
print(f"\nShard 00:")
print(f"  TF records parsed : {n_records:,}")
print(f"  Unique sequences  : {n_seqs:,}")
print(f"  Total frames      : {n_frames:,}")
print(f"  Mean frames/seq   : {n_frames/n_seqs:.1f}")


In [ ]:
# Frames per sequence distribution
fps = np.array([len(v) for v in seq_to_timestamps.values()])
print(f"Frames per sequence: min={fps.min()}  max={fps.max()}  "
      f"mean={fps.mean():.1f}  median={np.median(fps):.0f}")
print(f"Expected ~200 frames/seq for val (20s × 10Hz)")

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(fps, bins=30, color="#00C6C2", edgecolor="#0D1B2A", linewidth=0.5)
ax.set_xlabel("Frames per sequence")
ax.set_ylabel("# sequences")
ax.set_title(f"Shard 00: frames per sequence (n={n_seqs})")
plt.tight_layout()
plt.savefig("frames_per_seq.png", dpi=100, bbox_inches="tight")
plt.show()


In [ ]:
# Verify timestamps are in microseconds and extract time range
if PROTO_AVAILABLE:
    all_ts = np.array([ts for tss in seq_to_timestamps.values() for ts in tss])
    ts_sec = all_ts / 1e6  # microseconds → seconds

    import datetime
    dt_min = datetime.datetime.utcfromtimestamp(ts_sec.min())
    dt_max = datetime.datetime.utcfromtimestamp(ts_sec.max())
    print(f"Timestamp range (UTC):")
    print(f"  Earliest frame: {dt_min}")
    print(f"  Latest frame:   {dt_max}")
    print(f"  Total span:     {dt_max - dt_min}")
    print(f"\nMean inter-frame interval: {np.diff(np.sort(all_ts)).mean()/1e6*1000:.1f} ms  (expect 100ms = 10Hz)")

    fig, ax = plt.subplots(figsize=(8, 3))
    ax.hist(ts_sec, bins=50, color="#00C6C2", edgecolor="#0D1B2A", linewidth=0.5)
    ax.set_xlabel("Timestamp (Unix seconds)")
    ax.set_ylabel("Frame count")
    ax.set_title("Shard 00: frame timestamp distribution")
    plt.tight_layout()
    plt.savefig("timestamp_dist.png", dpi=100, bbox_inches="tight")
    plt.show()
else:
    print("Timestamps unavailable without proto library.")


## 4. Label Join: Sequences → Classes

Join shard sequences to the labels JSON using `frame.context.name` as the key.


In [ ]:
shard_labels = {}
unlabeled    = []
for seq_name in seq_to_timestamps:
    if seq_name in labels:
        shard_labels[seq_name] = labels[seq_name]["scenario_cluster"]
    else:
        unlabeled.append(seq_name)

print(f"Sequences in shard:       {n_seqs:,}")
print(f"  → matched in labels:    {len(shard_labels):,}")
print(f"  → NOT in labels:        {len(unlabeled):,}")
if unlabeled:
    print(f"  Sample unlabeled IDs: {unlabeled[:2]}")


In [ ]:
# Sequence-level class distribution
seq_cls = Counter(shard_labels.values())
print("Sequence-level class distribution (shard 00):\n")
for cls, cnt in sorted(seq_cls.items(), key=lambda x: -x[1]):
    pct = 100 * cnt / len(shard_labels)
    bar = "\u2588" * int(pct / 2)
    print(f"  {cls:<32s}  {cnt:3d}  ({pct:5.1f}%)  {bar}")


In [ ]:
# Frame-level class distribution
frame_cls = Counter()
for seq_name, cluster in shard_labels.items():
    frame_cls[cluster] += len(seq_to_timestamps[seq_name])

total_lf = sum(frame_cls.values())
print(f"Frame-level class distribution (total labeled frames = {total_lf:,}):\n")
for cls, cnt in sorted(frame_cls.items(), key=lambda x: -x[1]):
    pct = 100 * cnt / total_lf
    bar = "\u2588" * int(pct / 2)
    print(f"  {cls:<32s}  {cnt:5,}  ({pct:5.1f}%)  {bar}")


## 5. One Full Sequence — End-to-End Verified

Pull every frame for a single labeled sequence, display all 8 cameras at one timestep,
show a filmstrip of the front camera across the sequence, and print a full fact sheet.


In [ ]:
# Pick a labeled Cyclist sequence of median length
TARGET_CLASS = "Cyclist"
candidates = [s for s, c in shard_labels.items() if c == TARGET_CLASS]
if not candidates:
    TARGET_CLASS = sorted(seq_cls, key=lambda c: -seq_cls[c])[0]
    candidates = [s for s, c in shard_labels.items() if c == TARGET_CLASS]

by_len = sorted(candidates, key=lambda s: len(seq_to_timestamps[s]))
focus_seq = by_len[len(by_len) // 2]
focus_n   = len(seq_to_timestamps[focus_seq])

print(f"Focus sequence : {focus_seq}")
print(f"  Label        : {shard_labels[focus_seq]}")
print(f"  Frame count  : {focus_n}")


In [ ]:
# Extract ALL frames for this sequence
CAMERA_NAME_MAP = {1:"FRONT",2:"FRONT_LEFT",3:"FRONT_RIGHT",
                   4:"SIDE_LEFT",5:"SIDE_RIGHT",6:"REAR",
                   7:"REAR_LEFT",8:"REAR_RIGHT"}

# frames_data: list of (timestamp_micros, {cam_name: PIL.Image})
frames_data = []

if PROTO_AVAILABLE:
    for raw in dataset:
        fd = e2e_pb2.E2EDFrame()
        fd.ParseFromString(raw.numpy())
        if fd.frame.context.name != focus_seq:
            continue
        cams = {CAMERA_NAME_MAP.get(img.name, f"cam_{img.name}"):
                Image.open(io.BytesIO(img.image))
                for img in fd.frame.images}
        frames_data.append((fd.frame.timestamp_micros, cams))

    frames_data.sort(key=lambda x: x[0])
    print(f"Extracted {len(frames_data)} frames")
    print(f"Camera names in first frame: {sorted(frames_data[0][1].keys())}")

    # Verify 8 cameras
    cam_counts = Counter(len(f[1]) for f in frames_data)
    print(f"Cameras per frame: {dict(cam_counts)}")
    if set(cam_counts.keys()) == {8}:
        print("\u2713 All frames have exactly 8 cameras")

else:
    # Fallback: JPEG scan, cameras unnamed
    JPEG_SOI = b'\xff\xd8\xff'; JPEG_EOI = b'\xff\xd9'
    for raw in dataset:
        b = raw.numpy()
        ex = tf.train.Example()
        ex.ParseFromString(b)
        for k in ex.features.feature.keys():
            parts = k.rsplit("-", 1)
            if len(parts) == 2 and parts[0] == focus_seq:
                jpegs = []
                pos = 0
                while True:
                    s = b.find(JPEG_SOI, pos)
                    if s == -1: break
                    e = b.find(JPEG_EOI, s)
                    if e == -1: break
                    jpegs.append(Image.open(io.BytesIO(b[s:e+2])))
                    pos = e + 2
                cams = {f"cam_{i}": img for i, img in enumerate(jpegs)}
                frames_data.append((int(parts[1]), cams))
    frames_data.sort(key=lambda x: x[0])
    print(f"Extracted {len(frames_data)} frames (fallback mode — cameras unnamed)")


In [ ]:
# Display all 8 cameras from the middle frame of the sequence
mid_ts, mid_cams = frames_data[len(frames_data) // 2]

cam_order = ["FRONT_LEFT","FRONT","FRONT_RIGHT","SIDE_LEFT","SIDE_RIGHT",
             "REAR_LEFT","REAR","REAR_RIGHT"]
available_cams = [c for c in cam_order if c in mid_cams]
available_cams += [c for c in sorted(mid_cams.keys()) if c not in cam_order]

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()
for ax, cname in zip(axes, available_cams):
    img = mid_cams[cname]
    ax.imshow(img)
    ax.set_title(f"{cname}\n{img.width}\u00d7{img.height}", fontsize=9)
    ax.axis("off")
for ax in axes[len(available_cams):]:
    ax.axis("off")

import datetime
ts_str = str(datetime.datetime.utcfromtimestamp(mid_ts / 1e6)) if PROTO_AVAILABLE else str(mid_ts)
plt.suptitle(
    f"Sequence: {focus_seq[:16]}...  |  Label: {shard_labels[focus_seq]}\n"
    f"Frame timestamp: {ts_str} UTC",
    fontsize=10
)
plt.tight_layout()
plt.savefig("all_8_cameras_midframe.png", dpi=100, bbox_inches="tight")
plt.show()
print("Saved: all_8_cameras_midframe.png")


In [ ]:
# Front-camera filmstrip across the sequence
N = max(1, len(frames_data) // 8)
display_frames = frames_data[::N][:8]
front_key = "FRONT" if PROTO_AVAILABLE else "cam_0"

fig, axes = plt.subplots(1, len(display_frames), figsize=(3.5 * len(display_frames), 2.8))
if len(display_frames) == 1: axes = [axes]

for ax, (ts, cams) in zip(axes, display_frames):
    if front_key in cams:
        ax.imshow(cams[front_key])
    ax.set_title(f"t={ts}" if not PROTO_AVAILABLE else
                 f"{(ts - frames_data[0][0])/1e6:.1f}s", fontsize=8)
    ax.axis("off")

plt.suptitle(
    f"FRONT camera filmstrip — {focus_seq[:12]}...\n"
    f"Label: {shard_labels[focus_seq]}  |  {len(frames_data)} frames total",
    fontsize=10
)
plt.tight_layout()
plt.savefig("filmstrip_front_cam.png", dpi=100, bbox_inches="tight")
plt.show()


In [ ]:
# Confirmed structure for ONE sequence
img0 = list(frames_data[0][1].values())[0]
n_cams_confirmed = len(frames_data[0][1])
print("=" * 60)
print("ONE SEQUENCE — VERIFIED STRUCTURE")
print("=" * 60)
print(f"  context.name:         {focus_seq}")
print(f"  scenario_cluster:     {shard_labels.get(focus_seq, 'NOT IN LABELS')}")
print(f"  Frames extracted:     {len(frames_data)}")
print(f"  Cameras per frame:    {n_cams_confirmed}  (official: 8)")
print(f"  Total images:         {len(frames_data)} × {n_cams_confirmed} = {len(frames_data)*n_cams_confirmed}")
print(f"  Image resolution:     {img0.width} × {img0.height} px")
if PROTO_AVAILABLE:
    dur_s = (frames_data[-1][0] - frames_data[0][0]) / 1e6
    fps_actual = (len(frames_data) - 1) / dur_s if dur_s > 0 else 0
    print(f"  Sequence duration:    {dur_s:.2f} s  (expect ~20s for val)")
    print(f"  Frame rate:           {fps_actual:.1f} Hz  (expect 10 Hz)")


## 6. Shard Summary & Full-Dataset Projection

In [ ]:
n_cams_confirmed = 8  # verified above from proto

print("=" * 60)
print("SHARD 00 SUMMARY  (val_202504211843.tfrecord-00000-of-00093)")
print("=" * 60)
print(f"  File size:                {SHARD_PATH.stat().st_size / 1e9:.2f} GB")
print(f"  Unique sequences:         {n_seqs:,}")
print(f"    → labeled:              {len(shard_labels):,}")
print(f"    → unlabeled:            {len(unlabeled):,}")
print(f"  Total frames:             {n_frames:,}")
print(f"  Cameras per frame:        {n_cams_confirmed}  (FRONT, FRONT_LEFT, FRONT_RIGHT,")
print(f"                                     SIDE_LEFT, SIDE_RIGHT,")
print(f"                                     REAR, REAR_LEFT, REAR_RIGHT)")
print(f"  Total images in shard:    {n_frames * n_cams_confirmed:,}")
print()
print("  Class distribution (sequences in this shard):")
for cls, cnt in sorted(seq_cls.items(), key=lambda x: -x[1]):
    pct = 100 * cnt / len(shard_labels)
    print(f"    {cls:<32s}  {cnt:3d}  ({pct:.1f}%)")
print()
print("  Projection to full val set (479 sequences, 93 shards):")
print(f"    Est. total val frames:  ~{479 * 200:,}  (479 seqs × ~200 frames/seq)")
print(f"    Est. total val images:  ~{479 * 200 * 8:,}  (× 8 cameras)")
print(f"    Actual: run across all 93 shards to get exact numbers.")


In [ ]:
summary = {
    "shard": SHARD_PATH.name,
    "proto_used": PROTO_AVAILABLE,
    "n_sequences": n_seqs,
    "n_labeled_sequences": len(shard_labels),
    "n_unlabeled_sequences": len(unlabeled),
    "n_frames_total": n_frames,
    "cameras_per_frame": 8,
    "camera_names": ["FRONT","FRONT_LEFT","FRONT_RIGHT","SIDE_LEFT",
                     "SIDE_RIGHT","REAR","REAR_LEFT","REAR_RIGHT"],
    "n_images_total": n_frames * 8,
    "class_distribution_sequences": dict(seq_cls),
    "class_distribution_frames": dict(frame_cls),
}
out = DATA_DIR / "shard00_eda_summary.json"
with open(out, "w") as f:
    json.dump(summary, f, indent=2)
print(f"Summary saved → {out}")
